# Tutorial 03: Budget-Aware Compilation

Quantum hardware is expensive. Different backends have vastly different pricing -- from $0.00016/shot on IBM to $8.00/shot on Quantinuum. qb-compiler helps you make cost-conscious decisions:

1. Understanding vendor pricing with `VENDOR_PRICING` and `cost_per_shot()`
2. Estimating execution cost with `CostEstimator`
3. Optimizing within a budget with `BudgetOptimizer`
4. Using the `budget_optimal` compilation strategy
5. Finding the cheapest backend for your workload

By the end you will know how to compile and run circuits without surprise bills.

## 1. Vendor Pricing Overview

qb-compiler ships with per-shot pricing data for all supported backends.

In [ ]:
from qb_compiler.cost.pricing import VENDOR_PRICING, cost_per_shot, VendorPricing

In [ ]:
# Survey the cost landscape
print(f"{'Backend':20s}  {'Provider':12s}  {'$/shot':>12s}  {'1K shots':>10s}  {'10K shots':>10s}")
print("-" * 70)

for name, pricing in sorted(VENDOR_PRICING.items(), key=lambda x: x[1].cost_per_shot_usd):
    cps = pricing.cost_per_shot_usd
    print(
        f"{name:20s}  {pricing.provider:12s}  "
        f"${cps:>10.5f}  "
        f"${cps * 1000:>8.2f}  "
        f"${cps * 10000:>8.2f}"
    )

print()
print("Note the enormous range: IBM is ~50,000x cheaper per shot than Quantinuum.")
print("But Quantinuum's gate fidelity is also much higher, so fewer shots may suffice.")

In [ ]:
# Look up a specific backend's cost
ibm_cps = cost_per_shot("ibm_fez")
ionq_cps = cost_per_shot("ionq_aria")

print(f"IBM Fez:   ${ibm_cps:.5f}/shot")
print(f"IonQ Aria: ${ionq_cps:.2f}/shot")
print(f"IonQ is {ionq_cps / ibm_cps:.0f}x more expensive per shot than IBM.")

## 2. Cost Estimation with QBCompiler

The compiler's built-in `estimate_cost()` method factors in circuit depth and the backend's per-shot pricing to give a more accurate estimate.

In [ ]:
import math
from qb_compiler import QBCompiler, QBCircuit

In [ ]:
# Build a moderately deep circuit
circ = QBCircuit(4)
for layer in range(5):
    for q in range(4):
        circ.h(q)
    circ.cx(0, 1).cx(2, 3)
    circ.cx(1, 2)
    for q in range(4):
        circ.rz(q, math.pi / (layer + 2))
circ.measure_all()

print(f"Circuit: {circ}")
print(f"Depth:   {circ.depth}")
print(f"Gates:   {circ.gate_count}")

In [ ]:
# Compile for IBM Fez and estimate cost at different shot counts
compiler = QBCompiler.from_backend("ibm_fez")
result = compiler.compile(circ)
compiled = result.compiled_circuit

print(f"Compiled depth: {result.compiled_depth}")
print(f"Estimated fidelity: {result.estimated_fidelity:.4f}")
print()

print(f"{'Shots':>8s}  {'Cost':>10s}  {'Per-shot':>12s}  {'Within $1?':>10s}  {'Within $10?':>10s}")
print("-" * 55)
for shots in [100, 1_000, 10_000, 100_000, 1_000_000]:
    cost = compiler.estimate_cost(compiled, shots=shots)
    print(
        f"{shots:>8,}  "
        f"${cost.total_usd:>8.4f}  "
        f"${cost.cost_per_shot_usd:>10.6f}  "
        f"{'yes' if cost.within_budget(1.0) else 'NO':>10s}  "
        f"{'yes' if cost.within_budget(10.0) else 'NO':>10s}"
    )

### Comparing Cost Across Backends

The same circuit can have very different costs on different backends.

In [ ]:
shots = 4096

print(f"Cost comparison at {shots:,} shots:")
print(f"{'Backend':20s}  {'Depth':>6s}  {'Fidelity':>10s}  {'Cost':>10s}  {'$/fidelity':>12s}")
print("-" * 65)

for backend in ["ibm_fez", "ibm_torino", "rigetti_ankaa", "ionq_aria", "iqm_garnet"]:
    comp = QBCompiler.from_backend(backend)
    r = comp.compile(circ)
    cost = comp.estimate_cost(r.compiled_circuit, shots=shots)
    # Cost per unit fidelity -- lower is better
    cost_per_fid = cost.total_usd / max(r.estimated_fidelity, 0.001)
    print(
        f"{backend:20s}  "
        f"{r.compiled_depth:6d}  "
        f"{r.estimated_fidelity:10.4f}  "
        f"${cost.total_usd:>8.4f}  "
        f"${cost_per_fid:>10.4f}"
    )

## 3. BudgetOptimizer

The `BudgetOptimizer` finds the best compilation settings that fit within a given budget. It determines optimal shot count, recommends a strategy, and can even find the cheapest backend for your workload.

In [ ]:
from qb_compiler.cost.budget_optimizer import BudgetOptimizer, OptimizationResult

In [ ]:
optimizer = BudgetOptimizer(min_shots=100)

# What's the best I can do on IBM Fez with a $1 budget?
opt = optimizer.optimize(
    backend="ibm_fez",
    budget_usd=1.00,
    circuit_depth=50,
)

print(f"Result: {opt}")
print()
print(f"Backend:            {opt.backend}")
print(f"Recommended shots:  {opt.recommended_shots:,}")
print(f"Estimated fidelity: {opt.estimated_fidelity:.4f}")
print(f"Estimated cost:     ${opt.estimated_cost_usd:.4f}")
print(f"Recommended strat:  {opt.strategy}")
print(f"Notes:              {opt.notes}")

In [ ]:
# What if I only have $0.10?
opt_small = optimizer.optimize(
    backend="ibm_fez",
    budget_usd=0.10,
    circuit_depth=50,
)

print(f"$0.10 budget:")
print(f"  Recommended shots: {opt_small.recommended_shots:,}")
print(f"  Cost: ${opt_small.estimated_cost_usd:.4f}")
print(f"  Strategy: {opt_small.strategy}")

In [ ]:
# Compare different budgets on IonQ (expensive per shot)
print("IonQ Aria budget analysis:")
print(f"{'Budget':>8s}  {'Shots':>8s}  {'Cost':>10s}  {'Strategy':>18s}")
print("-" * 50)

for budget in [10.0, 50.0, 100.0, 500.0]:
    opt_ionq = optimizer.optimize(
        backend="ionq_aria",
        budget_usd=budget,
        circuit_depth=50,
    )
    print(
        f"${budget:>6.0f}  "
        f"{opt_ionq.recommended_shots:>8,}  "
        f"${opt_ionq.estimated_cost_usd:>8.2f}  "
        f"{opt_ionq.strategy:>18s}"
    )

### Budget Too Small

If the budget can't even afford the minimum number of shots, a `BudgetExceededError` is raised.

In [ ]:
from qb_compiler import BudgetExceededError

try:
    optimizer.optimize(
        backend="ionq_aria",
        budget_usd=1.00,  # $1 buys only ~3 shots on IonQ at $0.30/shot
    )
except BudgetExceededError as e:
    print(f"Budget exceeded: {e}")
    print(f"  Estimated cost: ${e.estimated_usd:.2f}")
    print(f"  Budget:         ${e.budget_usd:.2f}")
    print(f"  Shots needed:   {e.shots}")

## 4. Finding the Cheapest Backend

The `find_cheapest_backend()` method searches all known backends and recommends the one that fits your budget and qubit requirements.

In [ ]:
# Find the cheapest backend that can run 4096 shots for under $1
# and has at least 20 qubits
recommendation = optimizer.find_cheapest_backend(
    budget_usd=1.00,
    min_qubits=20,
    target_shots=4096,
)

if recommendation:
    print(f"Recommended backend: {recommendation.backend}")
    print(f"  Shots:    {recommendation.recommended_shots:,}")
    print(f"  Cost:     ${recommendation.estimated_cost_usd:.4f}")
    print(f"  Fidelity: {recommendation.estimated_fidelity:.4f}")
    print(f"  Strategy: {recommendation.strategy}")
else:
    print("No backend fits within the budget.")

In [ ]:
# Try with a larger budget to see more options
for budget in [0.50, 5.00, 100.00, 1000.00]:
    rec = optimizer.find_cheapest_backend(
        budget_usd=budget,
        min_qubits=5,
        target_shots=1000,
    )
    if rec:
        print(
            f"  Budget ${budget:>7.2f}  ->  {rec.backend:20s}  "
            f"fidelity={rec.estimated_fidelity:.3f}  "
            f"cost=${rec.estimated_cost_usd:.4f}"
        )
    else:
        print(f"  Budget ${budget:>7.2f}  ->  no backend fits")

## 5. Budget-Optimal Compilation Strategy

The `budget_optimal` strategy uses a lighter optimization pass pipeline (level 1) that compiles faster. It's best when per-shot cost is low and you can compensate for slightly lower fidelity with more shots.

In [ ]:
# Compare all three strategies on a circuit
circ = QBCircuit(4)
for q in range(4):
    circ.h(q)
circ.cx(0, 1).cx(1, 2).cx(2, 3)
for q in range(4):
    circ.rz(q, math.pi / 3)
circ.cx(0, 1).cx(1, 2).cx(2, 3)
circ.h(0).h(0)  # redundant pair
circ.cx(2, 3).cx(2, 3)  # redundant pair
circ.measure_all()

compiler_ibm = QBCompiler.from_backend("ibm_fez")
shots = 4096

print(f"Input: depth={circ.depth}, gates={circ.gate_count}")
print()
print(f"{'Strategy':20s}  {'Depth':>6s}  {'Gates':>6s}  {'Fidelity':>10s}  {'Compile ms':>10s}  {'Cost':>10s}")
print("-" * 70)

for strategy in ["fidelity_optimal", "depth_optimal", "budget_optimal"]:
    r = compiler_ibm.compile(circ, strategy=strategy)
    cost = compiler_ibm.estimate_cost(r.compiled_circuit, shots=shots)
    print(
        f"{strategy:20s}  "
        f"{r.compiled_depth:6d}  "
        f"{r.compiled_circuit.gate_count:6d}  "
        f"{r.estimated_fidelity:10.4f}  "
        f"{r.compilation_time_ms:10.2f}  "
        f"${cost.total_usd:>8.4f}"
    )

### Using the Budget Guard in compile()

The `compile()` method accepts a `budget_usd` parameter. If the estimated cost at 1024 shots exceeds this amount, compilation raises `BudgetExceededError` before returning.

In [ ]:
# This works -- the circuit is cheap on IBM
result = compiler_ibm.compile(circ, budget_usd=10.00)
print(f"Compiled successfully: fidelity={result.estimated_fidelity:.4f}")

# Now try on a very expensive backend (Quantinuum) with a tiny budget
compiler_q = QBCompiler.from_backend("quantinuum_h2")
try:
    compiler_q.compile(circ, budget_usd=0.50)
except BudgetExceededError as e:
    print(f"\nQuantinuum compilation blocked: {e}")

## 6. Practical Workflow: Budget-Constrained Experiment

Here's a complete workflow for running a quantum experiment within a fixed budget.

In [ ]:
EXPERIMENT_BUDGET_USD = 5.00
MIN_QUBITS = 4
TARGET_SHOTS = 4096

# Step 1: Find the best backend within budget
opt = optimizer.find_cheapest_backend(
    budget_usd=EXPERIMENT_BUDGET_USD,
    min_qubits=MIN_QUBITS,
    target_shots=TARGET_SHOTS,
)

if opt is None:
    print("No backend fits within budget. Consider reducing shot count.")
else:
    print(f"Step 1: Selected {opt.backend} (cost=${opt.estimated_cost_usd:.4f})")

    # Step 2: Compile using the recommended strategy
    comp = QBCompiler.from_backend(opt.backend, strategy=opt.strategy)
    result = comp.compile(circ, budget_usd=EXPERIMENT_BUDGET_USD)
    print(f"Step 2: Compiled (fidelity={result.estimated_fidelity:.4f}, depth={result.compiled_depth})")

    # Step 3: Final cost check
    final_cost = comp.estimate_cost(result.compiled_circuit, shots=opt.recommended_shots)
    remaining = EXPERIMENT_BUDGET_USD - final_cost.total_usd
    print(f"Step 3: Cost=${final_cost.total_usd:.4f}, remaining=${remaining:.4f}")

    # Step 4: Could we afford more shots with the remaining budget?
    cps = final_cost.cost_per_shot_usd
    extra_shots = int(remaining / cps) if cps > 0 else 0
    print(f"Step 4: Could run {extra_shots:,} additional shots with remaining budget")

## Summary

In this tutorial you learned:

- **Vendor pricing** -- `VENDOR_PRICING` and `cost_per_shot()` provide per-shot costs for all supported backends
- **Cost estimation** -- `QBCompiler.estimate_cost()` factors in circuit depth for more accurate estimates
- **CostEstimate** -- the result object with `total_usd`, `cost_per_shot_usd`, and `within_budget()`
- **BudgetOptimizer** -- `optimize()` finds optimal shots and strategy; `find_cheapest_backend()` recommends the best device
- **Budget guard** -- `compile(budget_usd=...)` blocks compilation if estimated cost exceeds the budget
- **Strategies** -- `budget_optimal` compiles faster at the cost of lighter optimization

Next up: [Tutorial 04 -- QEC-Aware Compilation (Preview)](04-qec-aware.ipynb) introduces what's coming in Phase 2.